In [1]:
import torch 
import tenseal as ts
import torch.nn as nn

**Creating a Dummy Model**

In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=2, out_features=3, bias=True)
        self.layer2 = nn.Linear(in_features=3, out_features=1, bias=True)

    def forward(self, x):
        res1 = self.layer1(x)
        res2 = self.layer2(res1)
        return res2

In [3]:
data = torch.tensor([1.0, 2.0])

In [4]:
model = MyModel()

In [5]:
logits = model(data)
logits

tensor([-0.6434], grad_fn=<ViewBackward0>)

**Accessing the model parameters in dictonary form**

In [6]:
model_state_dict = model.state_dict()

In [7]:
for name, param in model_state_dict.items():
    print(param.shape)

torch.Size([3, 2])
torch.Size([3])
torch.Size([1, 3])
torch.Size([1])


In [8]:
model_state_dict['layer1.weight']

tensor([[-0.2602,  0.1639],
        [ 0.6842,  0.5476],
        [-0.3818, -0.0381]])

**Homormorphic encryption and decryption**

In [9]:
from fhe_lib import fh_encryption as fhe
from fhe_lib import fh_decryption as fhd
from fhe_lib import utilities as util

import copy

In [10]:
class TensorEncrypted():
    def __init__(self, encrypted_tensor, original_shape):
        self.encrypted_tensor = encrypted_tensor
        self.original_shape = original_shape

In [11]:
context = fhe.generate_fhe_context()

In [12]:
secret_context = ts.context_from(fhe.get_secret_context_serialized(context))
public_context = ts.context_from(fhe.get_public_context_serialized(context))

In [13]:
model_state_encrypted = {}

In [14]:
# Encrypt the model parameters
for name, param in model_state_dict.items():
    original_shape = param.shape
    tensor_flattened = param.flatten().tolist()
    enc_tensor = ts.ckks_vector(public_context, tensor_flattened) 

    # Sample homomorphic calculations
    enc_tensor *= 2
    enc_tensor += 1
    
    enc_wrapped_tensor = TensorEncrypted(encrypted_tensor=enc_tensor, original_shape=original_shape)
    model_state_encrypted[name] = enc_wrapped_tensor

In [15]:
model_state_encrypted

{'layer1.weight': <__main__.TensorEncrypted at 0x24091d6f730>,
 'layer1.bias': <__main__.TensorEncrypted at 0x24091d6f100>,
 'layer2.weight': <__main__.TensorEncrypted at 0x24091d6f1c0>,
 'layer2.bias': <__main__.TensorEncrypted at 0x24091d6f280>}

**Optional: Perform homomorphic calculations**

In [16]:
model_state_decrypted = {}

In [17]:
# Decrypt the model parameters
for name, param in model_state_encrypted.items():
    decrypted_tensor_flattened = torch.tensor(fhd.decrypt_flattened_vector(secret_context, param.encrypted_tensor))
    decrypted_tensor = decrypted_tensor_flattened.reshape(param.original_shape)
    

    model_state_decrypted[name] = decrypted_tensor

**Compare the encrypted and decrypted model states**

In [18]:
model_state_dict

OrderedDict([('layer1.weight',
              tensor([[-0.2602,  0.1639],
                      [ 0.6842,  0.5476],
                      [-0.3818, -0.0381]])),
             ('layer1.bias', tensor([-0.5071, -0.5865,  0.4651])),
             ('layer2.weight', tensor([[-0.1128, -0.3756,  0.3193]])),
             ('layer2.bias', tensor([-0.2472]))])

In [19]:
model_state_decrypted

{'layer1.weight': tensor([[0.4795, 1.3277],
         [2.3685, 2.0952],
         [0.2364, 0.9239]]),
 'layer1.bias': tensor([-0.0141, -0.1729,  1.9302]),
 'layer2.weight': tensor([[0.7743, 0.2488, 1.6386]]),
 'layer2.bias': tensor([0.5056])}

In [20]:
loaded_model = MyModel()

In [21]:
loaded_model.load_state_dict(model_state_decrypted)

<All keys matched successfully>

**SUCCESS!**